# Task 5: Quantum Error Correction - Solution

**Qiskit 1.x Implementation**

## Learning Objectives

- Implement the 3-qubit bit-flip code
- Encode quantum information with redundancy
- Detect and correct single-qubit errors
- Use Toffoli (CCX) gates
- Compare error scenarios

## Estimated Time: 75-90 minutes

In [ ]:
# Google Colab Setup - Run this cell first if using Colab
import sys
if 'google.colab' in sys.modules:
    print("📦 Installing dependencies for Google Colab...")
    !pip install -q qiskit>=1.0.0 qiskit-aer>=0.13.0 matplotlib pylatexenc
    print("✓ Dependencies installed successfully!")
    print("You can now run the rest of the notebook.\n")
else:
    print("✓ Running in local environment")

## Setup and Imports

In [ ]:
# Import required libraries
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Qiskit 1.x imports
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

# Set random seed
np.random.seed(42)

# Configure matplotlib
style_path = Path('../../assets/styles/qcai_style.mplstyle')
if style_path.exists():
    plt.style.use(str(style_path))
else:
    plt.rcParams['figure.figsize'] = (12, 6)
    plt.rcParams['font.size'] = 12

print("Setup complete!")

# Check Qiskit version
import qiskit
print(f"Qiskit version: {qiskit.__version__}")

---

## Exercise 1: Encoding - Create Logical Qubit

### Background

The 3-qubit bit-flip code encodes:
- $|0_L\rangle = |000\rangle$
- $|1_L\rangle = |111\rangle$

For a general state $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$:

$$|\psi_L\rangle = \alpha|000\rangle + \beta|111\rangle$$

### Implementation

Use CNOT gates to create entanglement.

In [ ]:
# SOLUTION: Implement encoding circuit

def create_encoding_circuit(prepare_state_func=None):
    """
    Create 3-qubit encoding circuit.
    
    Args:
        prepare_state_func: Optional function to prepare initial state on qubit 0
    
    Returns:
        QuantumCircuit with encoding
    """
    # 3 data qubits, 2 syndrome qubits, 5 classical bits
    # Classical bits 0-1: for syndrome measurement
    # Classical bits 2-4: for data qubit measurement (when needed)
    qc = QuantumCircuit(5, 5)
    
    # Prepare initial state on qubit 0 (if provided)
    if prepare_state_func:
        prepare_state_func(qc)
    
    # Encoding: Create α|000⟩ + β|111⟩
    qc.cx(0, 1)  # Copy qubit 0 to qubit 1
    qc.cx(0, 2)  # Copy qubit 0 to qubit 2
    qc.barrier()
    
    return qc

# Test with |0⟩ state
qc_encode_0 = create_encoding_circuit()

print("Encoding circuit for |0⟩:")
print(f"  Qubits: {qc_encode_0.num_qubits}")
print(f"  Classical bits: {qc_encode_0.num_clbits}")
print(f"  Depth: {qc_encode_0.depth()}")

### Visualise Encoding Circuit

In [ ]:
# Draw encoding circuit
qc_encode_0.draw(output='mpl', style='iqp')
plt.show()

### Test Encoding with Different States

In [ ]:
# SOLUTION: Test encoding with different initial states

# Helper function to simulate and print
def simulate_and_show(qc, description, shots=1000):
    """
    Simulate circuit and display results.
    """
    # Add measurements (data qubits to classical bits 2, 3, 4)
    qc_meas = qc.copy()
    qc_meas.measure([0, 1, 2], [2, 3, 4])
    
    # Simulate using Qiskit 1.x
    simulator = AerSimulator()
    transpiled = transpile(qc_meas, simulator)
    job = simulator.run(transpiled, shots=shots)
    counts = job.result().get_counts()
    
    print(f"\n{description}:")
    for outcome, count in sorted(counts.items(), key=lambda x: x[1], reverse=True):
        percentage = (count / shots) * 100
        # Extract data bits (rightmost 3 bits are data qubits in bits 2,3,4)
        data_bits = outcome[-3:]
        print(f"  |{data_bits}⟩: {count:4d} ({percentage:5.2f}%)")
    
    return counts

# Test 1: Encode |0⟩
qc1 = create_encoding_circuit()
counts1 = simulate_and_show(qc1, "Encoding |0⟩ → |000⟩")

# Test 2: Encode |1⟩
def prep_one(qc):
    qc.x(0)

qc2 = create_encoding_circuit(prep_one)
counts2 = simulate_and_show(qc2, "Encoding |1⟩ → |111⟩")

# Test 3: Encode |+⟩
def prep_plus(qc):
    qc.h(0)

qc3 = create_encoding_circuit(prep_plus)
counts3 = simulate_and_show(qc3, "Encoding |+⟩ → (|000⟩+|111⟩)/√2")

**Expected:**
- |0⟩ → only |000⟩
- |1⟩ → only |111⟩
- |+⟩ → 50-50 mix of |000⟩ and |111⟩

---

## Exercise 2: Introduce Bit-Flip Error

### Background

A bit-flip error is simulated by applying an X gate to one of the data qubits.

### Implementation

In [ ]:
# SOLUTION: Add error to circuit

def add_error(qc, error_qubit):
    """
    Add bit-flip error to specified qubit.
    
    Args:
        qc: QuantumCircuit
        error_qubit: Which qubit to flip (0, 1, or 2)
    """
    if error_qubit is not None:
        qc.x(error_qubit)
    qc.barrier()
    return qc

# Create circuit with error on qubit 1
qc_with_error = create_encoding_circuit()
add_error(qc_with_error, error_qubit=1)

# Simulate
counts_error = simulate_and_show(qc_with_error, 
                                 "Encoded |0⟩ with error on qubit 1")

print("\nWithout error correction, state is corrupted!")

**Expected:** State changed from |000⟩ to |010⟩ (qubit 1 flipped)

---

## Exercise 3: Syndrome Measurement

### Background

Syndrome measurement detects which qubit flipped **without collapsing the encoded state**.

**Syndrome bits:**
- Qubit 3: Checks if qubits 0 and 1 are different
- Qubit 4: Checks if qubits 0 and 2 are different

### Implementation

In [ ]:
# SOLUTION: Implement syndrome measurement

def add_syndrome_measurement(qc):
    """
    Add syndrome measurement to circuit.
    
    Syndrome qubit 3: Checks qubits 0 and 1
    Syndrome qubit 4: Checks qubits 0 and 2
    """
    # Compare qubits 0 and 1 (result in qubit 3)
    qc.cx(0, 3)
    qc.cx(1, 3)
    
    # Compare qubits 0 and 2 (result in qubit 4)
    qc.cx(0, 4)
    qc.cx(2, 4)
    
    qc.barrier()
    
    # Measure syndrome qubits
    qc.measure(3, 0)
    qc.measure(4, 1)
    qc.barrier()
    
    return qc

# Test syndrome measurement with different errors
print("Testing syndrome measurement:\n")

for error_qubit in [None, 0, 1, 2]:
    qc_syndrome = create_encoding_circuit()
    add_error(qc_syndrome, error_qubit)
    add_syndrome_measurement(qc_syndrome)
    
    # Simulate
    simulator = AerSimulator()
    transpiled = transpile(qc_syndrome, simulator)
    job = simulator.run(transpiled, shots=1000)
    counts = job.result().get_counts()
    
    # Get syndrome (most common outcome)
    syndrome = max(counts, key=counts.get)
    
    error_desc = f"Error on qubit {error_qubit}" if error_qubit is not None else "No error"
    print(f"{error_desc:20s} → Syndrome: {syndrome}")

print("\nSyndrome interpretation:")
print("  00 → No error")
print("  11 → Qubit 0 flipped")
print("  10 → Qubit 1 flipped")
print("  01 → Qubit 2 flipped")

**Expected:** Each error produces unique syndrome

---

## Exercise 4: Error Correction

### Background

Based on the syndrome, apply corrections:
- Syndrome `11`: Flip qubit 0
- Syndrome `10`: Flip qubit 1
- Syndrome `01`: Flip qubit 2
- Syndrome `00`: No correction needed

### Implementation

In [ ]:
# SOLUTION: Implement error correction

def add_error_correction(qc):
    """
    Add error correction based on syndrome measurement.
    
    Uses classical control based on syndrome qubits 3 and 4.
    """
    # Correction based on syndrome
    # Note: We use controlled gates based on syndrome measurement
    
    # If both syndrome bits are 1 (syndrome 11), flip qubit 0
    qc.ccx(3, 4, 0)  # Toffoli: flip if both 3 and 4 are 1
    
    # If only syndrome bit 3 is 1 (syndrome 10), flip qubit 1
    # This requires checking: bit 3 = 1 AND bit 4 = 0
    # Equivalent: flip bit 1 controlled by bit 3, then unflip if bit 4 is 1
    qc.cx(3, 1)
    qc.ccx(3, 4, 1)  # Unflip if both are 1 (was syndrome 11, not 10)
    
    # If only syndrome bit 4 is 1 (syndrome 01), flip qubit 2
    qc.cx(4, 2)
    qc.ccx(3, 4, 2)  # Unflip if both are 1 (was syndrome 11, not 01)
    
    qc.barrier()
    return qc

# Create complete error correction circuit
def create_full_ec_circuit(prepare_func=None, error_qubit=None):
    """
    Create complete error correction circuit.
    
    Args:
        prepare_func: Function to prepare initial state
        error_qubit: Which qubit to flip (None for no error)
    """
    qc = create_encoding_circuit(prepare_func)
    add_error(qc, error_qubit)
    add_syndrome_measurement(qc)
    add_error_correction(qc)
    return qc

print("Complete error correction circuit created")

---

## Exercise 5: Test Error Correction

### Background

Verify that error correction recovers the original state.

### Implementation

In [ ]:
# SOLUTION: Test error correction for all error scenarios

print("Testing Error Correction:\n")
print("="*60)

for error_qubit in [None, 0, 1, 2]:
    print(f"\nError on qubit {error_qubit if error_qubit is not None else 'None'}:")
    print("-" * 60)
    
    # Create circuit with error correction
    qc_ec = create_full_ec_circuit(error_qubit=error_qubit)
    
    # Add final measurement of data qubits (to classical bits 2, 3, 4)
    qc_ec.measure([0, 1, 2], [2, 3, 4])
    
    # Simulate
    simulator = AerSimulator()
    transpiled = transpile(qc_ec, simulator)
    job = simulator.run(transpiled, shots=1000)
    counts = job.result().get_counts()
    
    # Display results (rightmost 3 bits are data qubits)
    print("  Data qubits after correction:")
    for outcome, count in sorted(counts.items(), key=lambda x: x[1], reverse=True)[:3]:
        # Extract data bits (rightmost 3 bits)
        data_bits = outcome[-3:]
        percentage = (count / 1000) * 100
        print(f"    |{data_bits}⟩: {count:4d} ({percentage:5.2f}%)")
    
print("\n" + "="*60)
print("✓ Error correction restores |000⟩ in all cases!")

**Expected:** All errors corrected, state restored to |000⟩

---

## Exercise 6: Visualise Complete Circuit

### Implementation

In [ ]:
# SOLUTION: Visualise complete error correction circuit

# Create circuit with error on qubit 1
qc_visual = create_full_ec_circuit(error_qubit=1)

# Draw circuit
qc_visual.draw(output='mpl', style='iqp', fold=25);
plt.suptitle("Complete 3-Qubit Error Correction Circuit (Error on Qubit 1)",
             fontsize=14, fontweight='bold', y=0.98)
plt.show()

print("\nCircuit sections:")
print("  1. Encoding (qubits 0-2)")
print("  2. Error introduction (X gate)")
print("  3. Syndrome measurement (qubits 3-4)")
print("  4. Error correction (based on syndrome)")

---

## Exercise 7: Test with Different Initial States

### Background

Error correction should work for **any** initial state, not just |0⟩.

### Implementation

In [ ]:
# SOLUTION: Test error correction with different initial states

print("Testing Error Correction with Different Initial States:\n")

test_cases = [
    ("| 0⟩", None),
    ("|1⟩", lambda qc: qc.x(0)),
    ("|+⟩", lambda qc: qc.h(0)),
    ("RY(π/4)|0⟩", lambda qc: qc.ry(np.pi/4, 0)),
]

for state_name, prep_func in test_cases:
    print(f"\nInitial state: {state_name}")
    print("-" * 40)
    
    # Without error correction
    qc_no_ec = create_encoding_circuit(prep_func)
    add_error(qc_no_ec, error_qubit=1)
    qc_no_ec.measure([0, 1, 2], [2, 3, 4])
    
    sim = AerSimulator()
    counts_no_ec = sim.run(transpile(qc_no_ec, sim), shots=1000).result().get_counts()
    
    print("  Without correction (error on qubit 1):")
    for outcome, count in sorted(counts_no_ec.items(), 
                                 key=lambda x: x[1], reverse=True)[:2]:
        data_bits = outcome[-3:]
        print(f"    |{data_bits}⟩: {count} times")
    
    # With error correction
    qc_with_ec = create_full_ec_circuit(prep_func, error_qubit=1)
    qc_with_ec.measure([0, 1, 2], [2, 3, 4])
    
    counts_with_ec = sim.run(transpile(qc_with_ec, sim), shots=1000).result().get_counts()
    
    print("  With correction:")
    for outcome, count in sorted(counts_with_ec.items(),
                                 key=lambda x: x[1], reverse=True)[:2]:
        data_bits = outcome[-3:]
        print(f"    |{data_bits}⟩: {count} times")

print("\n✓ Error correction works for arbitrary initial states!")

---

## Exercise 8: Compare Error Rates

### Implementation

In [ ]:
# SOLUTION: Compare error rates with and without correction

def calculate_error_rate(counts, expected_state):
    """
    Calculate error rate (fraction of outcomes != expected).
    """
    total = sum(counts.values())
    correct = counts.get(expected_state, 0)
    return 1 - (correct / total)

print("Error Rate Comparison:\n")

results = []

for error_qubit in [None, 0, 1, 2]:
    # Without correction
    qc_no = create_encoding_circuit()
    add_error(qc_no, error_qubit)
    qc_no.measure([0, 1, 2], [2, 3, 4])
    
    sim = AerSimulator()
    counts_no = sim.run(transpile(qc_no, sim), shots=1000).result().get_counts()
    
    # Extract data bits from outcomes
    counts_no_data = {}
    for outcome, count in counts_no.items():
        data_bits = outcome[-3:]
        counts_no_data[data_bits] = counts_no_data.get(data_bits, 0) + count
    
    error_rate_no = calculate_error_rate(counts_no_data, '000')
    
    # With correction
    qc_yes = create_full_ec_circuit(error_qubit=error_qubit)
    qc_yes.measure([0, 1, 2], [2, 3, 4])
    
    counts_yes = sim.run(transpile(qc_yes, sim), shots=1000).result().get_counts()
    
    # Extract data bits (rightmost 3 bits)
    counts_yes_data = {}
    for outcome, count in counts_yes.items():
        data_bits = outcome[-3:]
        counts_yes_data[data_bits] = counts_yes_data.get(data_bits, 0) + count
    
    error_rate_yes = calculate_error_rate(counts_yes_data, '000')
    
    error_desc = f"Qubit {error_qubit}" if error_qubit is not None else "None"
    results.append((error_desc, error_rate_no, error_rate_yes))
    
    print(f"Error on {error_desc:8s}:")
    print(f"  Without correction: {error_rate_no*100:5.1f}% error")
    print(f"  With correction:    {error_rate_yes*100:5.1f}% error")
    print()

print("✓ Error correction dramatically reduces error rates!")

### Visualise Error Rate Comparison

In [ ]:
# Visualise error rates
fig, ax = plt.subplots(figsize=(10, 6))

labels = [r[0] for r in results]
error_no_ec = [r[1] * 100 for r in results]
error_with_ec = [r[2] * 100 for r in results]

x = np.arange(len(labels))
width = 0.35

bars1 = ax.bar(x - width/2, error_no_ec, width, label='Without Correction',
              color='red', edgecolor='black', linewidth=1.5, alpha=0.7)
bars2 = ax.bar(x + width/2, error_with_ec, width, label='With Correction',
              color='green', edgecolor='black', linewidth=1.5, alpha=0.7)

ax.set_xlabel('Error Location', fontsize=12, fontweight='bold')
ax.set_ylabel('Error Rate (%)', fontsize=12, fontweight='bold')
ax.set_title('Error Correction Effectiveness', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend(fontsize=12)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---

## Exercise 9: Multiple Error Scenario

### Background

The 3-qubit code can only correct **one** error. What happens with two errors?

### Implementation

In [ ]:
# SOLUTION: Test with multiple errors

print("Testing Multiple Errors (Code Failure):\n")

# Create circuit with TWO errors
qc_multi = create_encoding_circuit()
qc_multi.x(0)  # Error 1
qc_multi.x(1)  # Error 2
qc_multi.barrier()

# Add syndrome measurement and correction
add_syndrome_measurement(qc_multi)
add_error_correction(qc_multi)

# Measure (data qubits to classical bits 2, 3, 4)
qc_multi.measure([0, 1, 2], [2, 3, 4])

# Simulate
sim = AerSimulator()
counts_multi = sim.run(transpile(qc_multi, sim), shots=1000).result().get_counts()

print("Errors on qubits 0 AND 1:")
for outcome, count in sorted(counts_multi.items(), key=lambda x: x[1], reverse=True)[:3]:
    data_bits = outcome[-3:]
    percentage = (count / 1000) * 100
    print(f"  |{data_bits}⟩: {count:4d} ({percentage:5.2f}%)")

print("\n⚠ With 2 errors, correction fails!")
print("   The 3-qubit code assumes at most 1 error.")
print("   For multiple errors, more sophisticated codes are needed.")

---

## Exercise 10: Summary and Statistics

### Implementation

In [ ]:
# SOLUTION: Display comprehensive summary

print("="*60)
print("QUANTUM ERROR CORRECTION - SUMMARY")
print("="*60)
print()

print("3-Qubit Bit-Flip Code:")
print("  Encoding: |ψ⟩ → α|000⟩ + β|111⟩")
print("  Syndrome Qubits: 2 (detect error location)")
print("  Correction: Apply X gate based on syndrome")
print()

print("Syndrome Table:")
print("  00 → No error")
print("  11 → Qubit 0 flipped")
print("  10 → Qubit 1 flipped")
print("  01 → Qubit 2 flipped")
print()

print("Key Insights:")
print("  ✓ Corrects single bit-flip errors")
print("  ✓ Works for arbitrary quantum states")
print("  ✓ Syndrome measurement doesn't collapse data")
print("  ✓ Uses entanglement, not cloning")
print("  ✗ Cannot correct 2+ simultaneous errors")
print()

print("Qiskit 1.x Features Used:")
print("  ✓ AerSimulator (not Aer.get_backend)")
print("  ✓ transpile() + run() workflow")
print("  ✓ No deprecated assemble() calls")
print("  ✓ Toffoli (CCX) gates for correction")
print()

# Circuit statistics
qc_stats = create_full_ec_circuit(error_qubit=1)
print("Circuit Statistics:")
print(f"  Total qubits: {qc_stats.num_qubits}")
print(f"  Classical bits: {qc_stats.num_clbits}")
print(f"  Circuit depth: {qc_stats.depth()}")
print(f"  Gate counts: {qc_stats.count_ops()}")
print()

print("="*60)
print("Error correction implementation complete! 🛡️")
print("="*60)

---

## Summary

### Key Concepts Mastered

✅ **Quantum Error Correction** - Protecting quantum information from errors

✅ **3-Qubit Encoding** - Creating logical qubits with redundancy

✅ **Syndrome Measurement** - Detecting errors without collapsing state

✅ **Error Correction** - Recovering from bit-flip errors

✅ **Toffoli Gates** - Multi-qubit controlled operations

✅ **Qiskit 1.x API** - Modern quantum programming

### Next Steps

1. Run tests: `pytest tests/test_task_05.py`
2. Explore documentation in `docs/`
3. Try extending to phase-flip errors
4. Research advanced codes (Shor, Steane, surface codes)

**Congratulations!** You've completed all quantum computing exercises! 🎊